# Análisis de Datos por Departamento — Colombia (4 años) — Versión R

Réplica exacta del notebook de Python en lenguaje **R** para **Google Colab**.

**Flujo del notebook:**
1. Configurar nombres de columnas
2. Subir los 4 CSV (2021–2024) por el panel de archivos de Colab
3. Seleccionar columnas a conservar
4. Unir DataFrames y crear variables nuevas
5. Estadística descriptiva y visualizaciones
6. Exportar resultado final

> **Importante**: este notebook usa el runtime **R** de Colab. Para activarlo, abra el notebook con la URL `https://colab.to/r` o en Colab cambie el runtime a R (Entorno de ejecución → Cambiar tipo de entorno → R).

In [ ]:
# ── Celda 1: Librerías ─────────────────────────────────────────────────────────
# En Colab R la mayoría de paquetes ya están preinstalados. Si faltara alguno,
# se instala con install.packages("<nombre>").

pkgs <- c("dplyr", "readr", "tidyr", "ggplot2", "stringi", "scales", "patchwork")
faltan <- setdiff(pkgs, rownames(installed.packages()))
if (length(faltan) > 0) install.packages(faltan, quiet = TRUE)

suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
  library(tidyr)
  library(ggplot2)
  library(stringi)
  library(scales)
  library(patchwork)
})

options(scipen = 999, dplyr.summarise.inform = FALSE)
theme_set(theme_minimal(base_size = 11))

cat("✓ Librerías cargadas\n")

---
## ⚙️ Celda 2 — CONFIGURACIÓN
**Edite aquí los nombres de las columnas según sus archivos CSV antes de continuar.**

In [ ]:
# ── Celda 2: Configuración ─────────────────────────────────────────────────────
#
# COLUMNA_DEPARTAMENTO: columna que identifica el departamento.
#   Si es numérico usa el código DANE (ej. 5 = Antioquia, 11 = Bogotá D.C.)
#   Si es texto usa el nombre del departamento (ej. 'ANTIOQUIA')
#
# Ejemplos comunes en datos ICFES / Saber Pro:
#   Código DANE  → 'ESTU_COD_DEPTO_PRESENTACION'
#   Nombre texto → 'ESTU_DEPTO_PRESENTACION'

COLUMNA_DEPARTAMENTO <- "ESTU_COD_DEPTO_PRESENTACION"   # ← CAMBIAR SI ES NECESARIO

# COLUMNA_RESULTADO: columna con el puntaje / resultado del examen.
# Ejemplos: 'MOD_RAZONA_CUANTITAT_PUNT', 'PUNT_GLOBAL', 'PUNTAJE_TOTAL'

COLUMNA_RESULTADO <- "MOD_RAZONA_CUANTITAT_PUNT"        # ← CAMBIAR SI ES NECESARIO

cat(sprintf("Columna departamento : %s\n", COLUMNA_DEPARTAMENTO))
cat(sprintf("Columna resultado    : %s\n", COLUMNA_RESULTADO))

---
## 📂 Celda 3 — Subir archivos CSV

En Colab, abra el **panel de archivos** (ícono de carpeta en la barra lateral izquierda) y **arrastre los 4 archivos `.csv` a la carpeta `/content/`** en el orden 2021 → 2022 → 2023 → 2024.

Una vez subidos, ejecute la celda siguiente para detectarlos automáticamente.

In [ ]:
# ── Celda 3: Detección de archivos cargados ────────────────────────────────────
# Detecta todos los .csv que el usuario haya subido a /content/

carpeta <- "/content"
if (!dir.exists(carpeta)) carpeta <- getwd()   # fallback fuera de Colab

archivos <- list.files(carpeta, pattern = "\\.csv$", full.names = TRUE,
                       ignore.case = TRUE)

# Orden por fecha de modificación (los más antiguos primero = orden de carga)
info <- file.info(archivos)
archivos <- archivos[order(info$mtime)]

cat(sprintf("📂 %d archivo(s) CSV detectado(s) en %s:\n\n", length(archivos), carpeta))
for (i in seq_along(archivos)) {
  cat(sprintf("   %d. %s\n", i, basename(archivos[i])))
}

if (length(archivos) < 4) {
  cat(sprintf("\n⚠ Se esperaban 4 archivos, se encontraron %d.\n", length(archivos)))
  cat("  Puede continuar con menos archivos pero las variables de año estarán incompletas.\n")
}

if (length(archivos) == 0) {
  stop("No hay archivos CSV en /content. Suba sus archivos al panel de archivos de Colab y vuelva a ejecutar esta celda.")
}

---
## 📋 Celda 4 — Crear DataFrames individuales por año

In [ ]:
# ── Celda 4: DataFrames individuales ───────────────────────────────────────────
#
# Los años se asignan en el mismo orden en que aparecieron los archivos.
# Si los subió en un orden diferente, edite la lista 'anios_orden'.

anios_orden <- c(2021, 2022, 2023, 2024)   # ← CAMBIAR SI EL ORDEN FUE DISTINTO

leer_csv <- function(ruta) {
  # readr::read_csv detecta separador y codificación; se prueban encodings comunes
  encs <- c("UTF-8", "latin1", "UTF-8-BOM", "CP1252")
  for (enc in encs) {
    df <- try(suppressWarnings(
      readr::read_csv(ruta, locale = readr::locale(encoding = enc),
                      show_col_types = FALSE, progress = FALSE)
    ), silent = TRUE)
    if (!inherits(df, "try-error") && ncol(df) > 1) return(as.data.frame(df))
  }
  stop(sprintf("No se pudo leer %s con ninguna codificación conocida.", ruta))
}

dataframes <- list()
for (i in seq_along(archivos)) {
  anio <- if (i <= length(anios_orden)) anios_orden[i] else 9000L + i
  df   <- leer_csv(archivos[i])
  df$AÑO <- anio
  dataframes[[as.character(anio)]] <- df
  cat(sprintf("✓ df_%d: %8s filas × %4d columnas  ← %s\n",
              anio, format(nrow(df), big.mark = ","), ncol(df), basename(archivos[i])))
}

# Variables individuales accesibles por nombre
df_2021 <- dataframes[["2021"]]
df_2022 <- dataframes[["2022"]]
df_2023 <- dataframes[["2023"]]
df_2024 <- dataframes[["2024"]]

cat("\n✓ DataFrames individuales creados: df_2021, df_2022, df_2023, df_2024\n")

---
## 🔧 Celda 5 — Seleccionar columnas a conservar

Edite el vector `columnas_a_conservar`. Tres modos de uso:

- **Conservar TODAS** → deje `columnas_a_conservar <- "TODAS"`.
- **Solo mínimas** → escriba `columnas_a_conservar <- "MINIMAS"`.
- **Selección personalizada** → escriba un vector con los nombres, p. ej. `c("FAMI_ESTRATOVIVIENDA", "ESTU_GENERO")`.

Las columnas `COLUMNA_DEPARTAMENTO`, `COLUMNA_RESULTADO` y `AÑO` se conservan siempre.

In [ ]:
# ── Celda 5: Selección de columnas ─────────────────────────────────────────────
df_ref <- dataframes[[1]]
todas_las_columnas <- setdiff(colnames(df_ref), "AÑO")

cat(sprintf("Total de columnas disponibles: %d\n", length(todas_las_columnas)))
cat(strrep("─", 50), "\n", sep = "")
for (i in seq_along(todas_las_columnas)) {
  cat(sprintf("  %4d. %s\n", i - 1L, todas_las_columnas[i]))
}

# ── EDITAR AQUÍ ───────────────────────────────────────────────────────────────
columnas_a_conservar <- "TODAS"   # "TODAS" / "MINIMAS" / c("col1", "col2", ...)
# ─────────────────────────────────────────────────────────────────────────────

if (identical(columnas_a_conservar, "TODAS")) {
  columnas_sel <- todas_las_columnas
} else if (identical(columnas_a_conservar, "MINIMAS")) {
  columnas_sel <- intersect(c(COLUMNA_DEPARTAMENTO, COLUMNA_RESULTADO), todas_las_columnas)
} else {
  columnas_sel <- intersect(columnas_a_conservar, todas_las_columnas)
}

# Garantizar columnas necesarias
for (req in c(COLUMNA_DEPARTAMENTO, COLUMNA_RESULTADO)) {
  if (req %in% todas_las_columnas && !(req %in% columnas_sel)) {
    columnas_sel <- c(columnas_sel, req)
  }
}

cat(sprintf("\n✓ Confirmadas %d columnas:\n", length(columnas_sel)))
print(columnas_sel)

---
## 🔗 Celda 6 — Unir los 4 DataFrames

In [ ]:
# ── Celda 6: Unión de los 4 DataFrames ─────────────────────────────────────────
cols_keep <- c(columnas_sel, "AÑO")

filtrar <- function(df, cols) {
  cols_existentes <- intersect(cols, colnames(df))
  df[, cols_existentes, drop = FALSE]
}

piezas <- lapply(dataframes, filtrar, cols = cols_keep)
df_total <- dplyr::bind_rows(piezas)

cat(sprintf("✓ DataFrame unificado: %s filas × %d columnas\n",
            format(nrow(df_total), big.mark = ","), ncol(df_total)))
cat("  Columnas:\n")
print(colnames(df_total))
head(df_total, 3)

---
## ➕ Celda 7 — Crear variables nuevas
Se crean cuatro variables:

| Variable | Descripción |
|---|---|
| `DUMMY_POST` | 0 si el registro es de 2021 o 2022, 1 si es de 2023 o 2024 |
| `NUM_DEPTO` | Número del departamento (1–32), **excluye Bogotá D.C.** |
| `DIST_BOGOTA_KM` | Distancia por carretera desde la capital del depto hasta Bogotá |
| `PROM_RESULTADO_DEPTO` | Promedio del resultado del examen para todos los registros del departamento |

In [ ]:
# ── Celda 7: Creación de variables nuevas ──────────────────────────────────────

# ── 7.1  DUMMY_POST ───────────────────────────────────────────────────────────
# 0 = Pre (2021 / 2022)   |   1 = Post (2023 / 2024)
df_total$DUMMY_POST <- ifelse(df_total$AÑO %in% c(2021, 2022), 0L, 1L)

cat("✓ DUMMY_POST (0 = 2021-2022 / 1 = 2023-2024):\n")
tab_dummy <- table(factor(df_total$DUMMY_POST, levels = c(0, 1),
                          labels = c("Pre  (2021-2022)", "Post (2023-2024)")))
print(tab_dummy)

# ── 7.2  NUM_DEPTO  ───────────────────────────────────────────────────────────
# Mapeo código DANE → número 1-32 (Bogotá D.C. → NA, excluido del análisis)
MAPA_DANE <- c(
  "91" = 1L,   #  1 Amazonas
  "5"  = 2L,   #  2 Antioquia
  "81" = 3L,   #  3 Arauca
  "8"  = 4L,   #  4 Atlántico
  "13" = 5L,   #  5 Bolívar
  "15" = 6L,   #  6 Boyacá
  "17" = 7L,   #  7 Caldas
  "18" = 8L,   #  8 Caquetá
  "85" = 9L,   #  9 Casanare
  "19" = 10L,  # 10 Cauca
  "20" = 11L,  # 11 Cesar
  "27" = 12L,  # 12 Chocó
  "23" = 13L,  # 13 Córdoba
  "25" = 14L,  # 14 Cundinamarca
  "94" = 15L,  # 15 Guainía
  "95" = 16L,  # 16 Guaviare
  "41" = 17L,  # 17 Huila
  "44" = 18L,  # 18 La Guajira
  "47" = 19L,  # 19 Magdalena
  "50" = 20L,  # 20 Meta
  "52" = 21L,  # 21 Nariño
  "54" = 22L,  # 22 Norte de Santander
  "86" = 23L,  # 23 Putumayo
  "63" = 24L,  # 24 Quindío
  "66" = 25L,  # 25 Risaralda
  "88" = 26L,  # 26 San Andrés y Providencia
  "68" = 27L,  # 27 Santander
  "70" = 28L,  # 28 Sucre
  "73" = 29L,  # 29 Tolima
  "76" = 30L,  # 30 Valle del Cauca
  "97" = 31L,  # 31 Vaupés
  "99" = 32L,  # 32 Vichada
  "11" = NA_integer_   # Bogotá D.C. — EXCLUIDO
)

# Mapeo nombre en texto (mayúsculas, sin tildes) → número 1-32
MAPA_NOMBRE <- c(
  "AMAZONAS" = 1L, "ANTIOQUIA" = 2L, "ARAUCA" = 3L,
  "ATLANTICO" = 4L, "BOLIVAR" = 5L, "BOYACA" = 6L,
  "CALDAS" = 7L, "CAQUETA" = 8L, "CASANARE" = 9L,
  "CAUCA" = 10L, "CESAR" = 11L, "CHOCO" = 12L,
  "CORDOBA" = 13L, "CUNDINAMARCA" = 14L,
  "GUAINIA" = 15L, "GUAVIARE" = 16L,
  "HUILA" = 17L, "LA GUAJIRA" = 18L, "GUAJIRA" = 18L,
  "MAGDALENA" = 19L, "META" = 20L,
  "NARINO" = 21L,
  "NORTE DE SANTANDER" = 22L, "PUTUMAYO" = 23L,
  "QUINDIO" = 24L, "RISARALDA" = 25L,
  "SAN ANDRES" = 26L, "SAN ANDRES Y PROVIDENCIA" = 26L,
  "SANTANDER" = 27L, "SUCRE" = 28L, "TOLIMA" = 29L,
  "VALLE DEL CAUCA" = 30L, "VALLE" = 30L,
  "VAUPES" = 31L, "VICHADA" = 32L,
  # Bogotá D.C. en sus distintas variantes → EXCLUIDA
  "BOGOTA" = NA_integer_, "BOGOTA D.C." = NA_integer_,
  "BOGOTA D.C" = NA_integer_, "DISTRITO CAPITAL" = NA_integer_
)

col <- df_total[[COLUMNA_DEPARTAMENTO]]
if (is.numeric(col)) {
  df_total$NUM_DEPTO <- MAPA_DANE[as.character(col)]
  tipo_mapa <- "código DANE numérico"
} else {
  col_norm <- toupper(stringi::stri_trans_general(as.character(col), "Latin-ASCII"))
  col_norm <- trimws(col_norm)
  df_total$NUM_DEPTO <- MAPA_NOMBRE[col_norm]
  tipo_mapa <- "nombre de departamento"
}
df_total$NUM_DEPTO <- unname(df_total$NUM_DEPTO)

cat(sprintf("\n✓ NUM_DEPTO creada (método: %s)\n", tipo_mapa))
cat(sprintf("  Registros sin mapeo (incl. Bogotá D.C.): %s\n",
            format(sum(is.na(df_total$NUM_DEPTO)), big.mark = ",")))

# ── 7.3  DIST_BOGOTA_KM ──────────────────────────────────────────────────────
# Distancia por carretera desde la capital departamental hasta Bogotá (km).
# Fuente: estimaciones estándar IGAC / INVIAS (vías terrestres).
DIST_BOGOTA <- c(
  "1" = 1500,  #  1 Amazonas    — Leticia         (ruta mixta)
  "2" =  415,  #  2 Antioquia   — Medellín
  "3" =  800,  #  3 Arauca      — Arauca
  "4" = 1002,  #  4 Atlántico   — Barranquilla
  "5" = 1044,  #  5 Bolívar     — Cartagena
  "6" =  148,  #  6 Boyacá      — Tunja
  "7" =  308,  #  7 Caldas      — Manizales
  "8" =  487,  #  8 Caquetá     — Florencia
  "9" =  360,  #  9 Casanare    — Yopal
  "10" =  594, # 10 Cauca       — Popayán
  "11" = 1011, # 11 Cesar       — Valledupar
  "12" =  529, # 12 Chocó       — Quibdó
  "13" =  821, # 13 Córdoba     — Montería
  "14" =    0, # 14 Cundinamarca— Bogotá (capital departamental = Bogotá)
  "15" = 1800, # 15 Guainía     — Inírida         (ruta mixta)
  "16" =  679, # 16 Guaviare    — San José del Guaviare
  "17" =  303, # 17 Huila       — Neiva
  "18" = 1204, # 18 La Guajira  — Riohacha
  "19" = 1096, # 19 Magdalena   — Santa Marta
  "20" =   89, # 20 Meta        — Villavicencio
  "21" =  869, # 21 Nariño      — Pasto
  "22" =  605, # 22 Nte. Santander — Cúcuta
  "23" =  612, # 23 Putumayo    — Mocoa
  "24" =  291, # 24 Quindío     — Armenia
  "25" =  341, # 25 Risaralda   — Pereira
  "26" =  752, # 26 San Andrés  — San Andrés (distancia aérea)
  "27" =  396, # 27 Santander   — Bucaramanga
  "28" =  942, # 28 Sucre       — Sincelejo
  "29" =  204, # 29 Tolima      — Ibagué
  "30" =  462, # 30 Valle Cauca — Cali
  "31" = 1600, # 31 Vaupés      — Mitú             (ruta mixta)
  "32" = 1395  # 32 Vichada     — Puerto Carreño
)
df_total$DIST_BOGOTA_KM <- unname(DIST_BOGOTA[as.character(df_total$NUM_DEPTO)])
cat("\n✓ DIST_BOGOTA_KM creada\n")

# ── 7.4  PROM_RESULTADO_DEPTO ─────────────────────────────────────────────────
# Media del resultado del examen por departamento, asignada a cada registro.
df_total <- df_total %>%
  dplyr::group_by(.data[[COLUMNA_DEPARTAMENTO]]) %>%
  dplyr::mutate(PROM_RESULTADO_DEPTO = mean(.data[[COLUMNA_RESULTADO]], na.rm = TRUE)) %>%
  dplyr::ungroup() %>%
  as.data.frame()
cat("✓ PROM_RESULTADO_DEPTO creada\n")

# ── Resumen ───────────────────────────────────────────────────────────────────
cat("\n", strrep("─", 55), "\n", sep = "")
cat(sprintf("%-25s %12s %10s\n", "Variable", "No nulos", "Nulos"))
cat(strrep("─", 55), "\n", sep = "")
for (v in c("DUMMY_POST", "NUM_DEPTO", "DIST_BOGOTA_KM", "PROM_RESULTADO_DEPTO")) {
  nn <- sum(!is.na(df_total[[v]]))
  nl <- sum( is.na(df_total[[v]]))
  cat(sprintf("%-25s %12s %10s\n", v,
              format(nn, big.mark = ","), format(nl, big.mark = ",")))
}

---
## 🗑️ Celda 8 — Excluir Bogotá D.C. y registros sin mapeo

In [ ]:
# ── Celda 8: Excluir Bogotá D.C. ───────────────────────────────────────────────
n_antes <- nrow(df_total)
df_total <- df_total[!is.na(df_total$NUM_DEPTO), , drop = FALSE]
df_total$NUM_DEPTO <- as.integer(df_total$NUM_DEPTO)
n_despues <- nrow(df_total)

cat(sprintf("Registros eliminados (Bogotá D.C. + sin mapeo): %s\n",
            format(n_antes - n_despues, big.mark = ",")))
cat(sprintf("Registros finales para análisis               : %s\n",
            format(n_despues, big.mark = ",")))

# Tabla de referencia: número ↔ nombre ↔ distancia
NOMBRES_DEPTO <- c(
  "Amazonas", "Antioquia", "Arauca", "Atlántico", "Bolívar", "Boyacá",
  "Caldas", "Caquetá", "Casanare", "Cauca", "Cesar", "Chocó",
  "Córdoba", "Cundinamarca", "Guainía", "Guaviare", "Huila", "La Guajira",
  "Magdalena", "Meta", "Nariño", "Norte de Santander", "Putumayo", "Quindío",
  "Risaralda", "San Andrés y Prov.", "Santander", "Sucre", "Tolima",
  "Valle del Cauca", "Vaupés", "Vichada"
)
df_total$NOMBRE_DEPTO <- NOMBRES_DEPTO[df_total$NUM_DEPTO]

cat("\n✓ Tabla de referencia departamentos:\n")
ref <- data.frame(
  Num = 1:32,
  Departamento = NOMBRES_DEPTO,
  Capital = c("Leticia", "Medellín", "Arauca", "Barranquilla", "Cartagena",
              "Tunja", "Manizales", "Florencia", "Yopal", "Popayán",
              "Valledupar", "Quibdó", "Montería", "Bogotá", "Inírida",
              "S.J. Guaviare", "Neiva", "Riohacha", "Santa Marta", "Villavicencio",
              "Pasto", "Cúcuta", "Mocoa", "Armenia", "Pereira",
              "San Andrés", "Bucaramanga", "Sincelejo", "Ibagué", "Cali",
              "Mitú", "Puerto Carreño"),
  Dist_Bogota_km = unname(DIST_BOGOTA[as.character(1:32)])
)
print(ref, row.names = FALSE)

---
## 📊 Celda 9 — Estadística descriptiva

In [ ]:
# ── Celda 9: Estadística descriptiva ───────────────────────────────────────────
vars_analisis <- c(COLUMNA_RESULTADO, "DUMMY_POST", "NUM_DEPTO",
                   "DIST_BOGOTA_KM", "PROM_RESULTADO_DEPTO")
vars_ok <- intersect(vars_analisis, colnames(df_total))

cat(strrep("=", 70), "\n", sep = "")
cat("  ESTADÍSTICA DESCRIPTIVA — VARIABLES PRINCIPALES\n")
cat(strrep("=", 70), "\n", sep = "")
print(summary(df_total[, vars_ok, drop = FALSE]))

# ── Distribución por año ──────────────────────────────────────────────────────
cat("\n--- Registros por año ---\n")
tab_anio <- as.data.frame(table(AÑO = df_total$AÑO))
names(tab_anio)[2] <- "N_registros"
print(tab_anio, row.names = FALSE)

# ── Distribución por periodo ──────────────────────────────────────────────────
cat("\n--- Registros por periodo (DUMMY_POST) ---\n")
tab_per <- as.data.frame(table(
  Periodo = factor(df_total$DUMMY_POST, levels = c(0, 1),
                   labels = c("Pre  (2021-2022)", "Post (2023-2024)"))
))
names(tab_per)[2] <- "N_registros"
print(tab_per, row.names = FALSE)

# ── Promedio del resultado por departamento ───────────────────────────────────
cat("\n--- Promedio del resultado por departamento ---\n")
tabla <- df_total %>%
  dplyr::group_by(NUM_DEPTO, NOMBRE_DEPTO) %>%
  dplyr::summarise(
    Promedio = round(mean(.data[[COLUMNA_RESULTADO]], na.rm = TRUE), 3),
    Desv_Est = round(sd(  .data[[COLUMNA_RESULTADO]], na.rm = TRUE), 3),
    N        = dplyr::n(),
    .groups  = "drop"
  ) %>%
  dplyr::arrange(NUM_DEPTO) %>%
  as.data.frame()
print(tabla, row.names = FALSE)

---
## 📈 Celda 10 — Visualizaciones

In [ ]:
# ── Celda 10: Visualizaciones ──────────────────────────────────────────────────
media_gral <- mean(df_total[[COLUMNA_RESULTADO]], na.rm = TRUE)

# ── Panel 1: Histograma del puntaje ───────────────────────────────────────────
p1 <- ggplot(df_total, aes(x = .data[[COLUMNA_RESULTADO]])) +
  geom_histogram(bins = 60, fill = "#2196F3", color = "white", alpha = 0.85) +
  geom_vline(xintercept = media_gral, color = "red", linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = media_gral, y = Inf,
           label = sprintf("Media: %.1f", media_gral),
           color = "red", hjust = -0.1, vjust = 1.5, size = 3.5) +
  labs(title = "Distribución del Resultado del Examen",
       x = "Puntaje", y = "Frecuencia")

# ── Panel 2: Boxplot por año ───────────────────────────────────────────────────
p2 <- ggplot(df_total, aes(x = factor(AÑO), y = .data[[COLUMNA_RESULTADO]],
                            fill = factor(AÑO))) +
  geom_boxplot(outlier.size = 0.5, outlier.alpha = 0.4) +
  scale_fill_manual(values = c("#90CAF9", "#42A5F5", "#FB8C00", "#EF5350")) +
  labs(title = "Resultado por Año", x = "Año", y = "Puntaje") +
  theme(legend.position = "none")

# ── Panel 3: Promedio por departamento (barra horizontal) ─────────────────────
prom_depto <- df_total %>%
  dplyr::group_by(NUM_DEPTO, NOMBRE_DEPTO) %>%
  dplyr::summarise(promedio = mean(.data[[COLUMNA_RESULTADO]], na.rm = TRUE),
                   .groups = "drop") %>%
  dplyr::mutate(NOMBRE_DEPTO = factor(NOMBRE_DEPTO,
                                       levels = NOMBRE_DEPTO[order(promedio)]),
                color = ifelse(promedio < media_gral, "bajo", "alto"))

p3 <- ggplot(prom_depto, aes(x = promedio, y = NOMBRE_DEPTO, fill = color)) +
  geom_col(alpha = 0.85, color = "white") +
  geom_vline(xintercept = media_gral, color = "black", linetype = "dashed", linewidth = 0.6) +
  scale_fill_manual(values = c("bajo" = "#EF5350", "alto" = "#66BB6A")) +
  labs(title = "Promedio por Departamento\n(rojo = bajo la media nacional)",
       x = "Promedio", y = NULL) +
  theme(legend.position = "none", axis.text.y = element_text(size = 7))

# ── Panel 4: Dispersión distancia vs promedio ─────────────────────────────────
agg <- df_total %>%
  dplyr::group_by(NUM_DEPTO) %>%
  dplyr::summarise(promedio  = mean(.data[[COLUMNA_RESULTADO]], na.rm = TRUE),
                   distancia = dplyr::first(DIST_BOGOTA_KM),
                   nombre    = dplyr::first(NOMBRE_DEPTO),
                   .groups   = "drop")

p4 <- ggplot(agg, aes(x = distancia, y = promedio)) +
  geom_point(size = 3, color = "#7B1FA2", alpha = 0.85) +
  geom_smooth(method = "lm", se = FALSE, color = "red",
              linetype = "dashed", linewidth = 0.7, formula = y ~ x) +
  geom_text(aes(label = NUM_DEPTO), vjust = -0.9, size = 2.5, color = "#333") +
  labs(title = "Distancia a Bogotá vs Promedio del Resultado",
       x = "Distancia a Bogotá (km)", y = "Promedio del Resultado")

# ── Combinar paneles con patchwork ────────────────────────────────────────────
figura <- (p1 | p2) / (p3 | p4) +
  patchwork::plot_annotation(
    title = "Análisis Descriptivo — Resultados del Examen por Departamento",
    theme = theme(plot.title = element_text(face = "bold", size = 14, hjust = 0.5))
  )

ggsave("analisis_departamentos.png", figura, width = 16, height = 12,
       dpi = 150, bg = "white")
print(figura)
cat("✓ Gráfico guardado como analisis_departamentos.png\n")

---
## 💾 Celda 11 — Exportar DataFrame final

In [ ]:
# ── Celda 11: Exportar ─────────────────────────────────────────────────────────
nombre_salida <- "datos_analisis_departamentos.csv"
readr::write_csv(df_total, nombre_salida)

# Descarga automática en Colab (usa el bridge Python detrás de R)
descargado <- tryCatch({
  system(sprintf("python3 -c \"from google.colab import files; files.download('%s')\"",
                 nombre_salida), intern = FALSE) == 0
}, error = function(e) FALSE, warning = function(w) FALSE)

if (!isTRUE(descargado)) {
  cat(sprintf("ℹ Descargue el archivo manualmente desde el panel de archivos: %s\n",
              nombre_salida))
}

cat("✓ Proceso completado.\n")
cat(sprintf("  Archivo exportado : %s\n", nombre_salida))
cat(sprintf("  Registros finales : %s\n", format(nrow(df_total), big.mark = ",")))
cat(sprintf("  Columnas totales  : %d\n", ncol(df_total)))
cat("\nVariables nuevas incluidas:\n")
cat("  AÑO                  — Año del registro (2021-2024)\n")
cat("  DUMMY_POST           — 0=Pre (2021-2022) / 1=Post (2023-2024)\n")
cat("  NUM_DEPTO            — Número del departamento (1-32)\n")
cat("  NOMBRE_DEPTO         — Nombre del departamento\n")
cat("  DIST_BOGOTA_KM       — Distancia capital depto → Bogotá (km)\n")
cat("  PROM_RESULTADO_DEPTO — Promedio del resultado para el departamento\n")